In [2]:
import polars as pl
from pathlib import Path
import jellyfish
from nltk.corpus import names
import nltk
#nltk.download('names')
import re
import altair as alt

In [5]:
samples_df = pl.read_parquet("jails_pdfs_presentation.parquet")

In [ ]:
yes = samples_df.filter(
    pl.col("Resulting Death?").str.contains("Yes")
)

In [6]:
samples_df

Deceased Name,Deceased Cause,Deceased Date and Time,Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness,Facility Name,Facility Type,RD Number,Phone Number,Address,Date,Time of Day,AM or PM,Occurrence,Table Contents,Injuries?,Resulting Death?,process,Report ID,OCR_Confidence,OCR_Word_Count,OCR_Text_Detected,Cook County?
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,list[str],list[str],str,str,str,str,f64,i64,bool,str
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""‘acility Name: Kane County She…","""County ""","""R.D. Number: ""","""__ Telephone #: 630-232-6840 ""","""\ddress: 37W755 IL Route 38 Su…","""Date of Occurrence: 01/17/2021…","""_ Time of Occurrence: 193\ ""","""No filled box found""","[""Other (specify): Resisting "", ""Restraints Used ""]","[""Ponce, David Julian"", ""04/13/1999"", … ""Crim Damage,DUI (X2),Resisting""]","""No injuries""","""No filled box found""","""normal""","""Ponce, David 1-17_p1""",89.921053,304,true,null
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""Facility Name: Sangamon County…","""County ""","""R.D. Number: S _ = = ""","""\y a _. Telephone #: 717-753-6…","""Address: #1 Sheriffs Plaza Spr…","""Date of Occurrence: } — IG ~ 2…","""_ Time of Occurrence: O?7 40 ""","""aT. """,[],"[""Madk In, Ds iW..."", ""Pag i""]","""No injuries""","""Error: less than two boxes fou…","""normal""","""Macklin, Dion 1-16_p1""",84.018576,323,true,null
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""-acility Name: St. Clair Count…","""Error creating ROI: zero-lengt…","""R.D. Number: ; ""","""‘elephone #: 618-277-3505 ""","""\ddress: 700 North 5"" Street B…","""Date of Occurrence: 01/02/2021…","""me of Occurrence: 11:10 Oam. ""","""Error: less than two boxes fou…","[""Restraints Used —[ ""]","[""| 12/21/1995"", ""| 01/02/2021"", … ""Kiger, Bonham""]","""No injuries""","""No ""","""normal""","""Kiger, Bonham 1-2_p1""",87.941718,326,true,null
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""-acility Name: Tazewell County…","""County ""","""R.D.Number: ""","""_._ Telephone #: 309-478-5613 ""","""\ddress: 101 S. Capitol Pekin …","""Date of Occurrence: 1/10/2020 ""","""_ Time of Occurrence: 0435 ""","""a.m. ""","[""Restraints Used "", ""Other (specify): Control Used ""]","[""01/28/200 1"", ""- o1/ior2021"", … ""Domestic Battery""]","""No injuries""","""No filled box found""","""normal""","""Martin, Kaitlyn 1-10_p1""",88.321192,302,true,null
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""‘acility Name: Knox County She…","""Error creating ROI: text ROI i…","""R.D, Number: KN21-0095 ""","""____ Telephone #: 309-345-3760…","""iddress: 152 S. Kellogg St. Ga…","""Shoo Date of Occurrence: 01/05…","""wy wee _ Time of Occurrence: 1…","""pM. ""","[""Restraints Used "", ""Sther (spocify): self harm ""]","[""aera seater"", ""Whiting, Preston Scott"", … ""cette""]","""No injuries""","""No ""","""normal""","""Whiting, Preston 1-5_p1""",82.486301,292,true,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""‘acility Name: Cook County Dep…","""Caunty ""","""R.D. Number: DIV08-2023-17030 ""","""Telephone #: 773.674.3474 O IL…","""\ddress: 2700 S. California Di…","""Date of Occurrence: 7/31/2023 …","""of Occurrence: 9:16:00 PM ""","""Check time, no AM or PM for Co…","[""Fighting among ""]","[""Earnest R Rupert"", ""6/26/1980"", … ""7/28/2023""]","""No injuries""","""No filled box found""","""normal""","""FOIA - UO July 2023_p564""",88.549296,284,true,"""Cook County"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""-acility Name: Will County Adu…","""County ""","""R.D. Number; ""","""""","""address; QO ChicagoSt ———s—__…","""erect Date of Occurrence: MBW2…","""Why vial _ Time of Occurrence:…","""a.m. ""","[""Assault on Staff ""]","[""ae ee"", ""-Deiainees

In [195]:
processing = pl.read_parquet("processing_logs_full.parquet")

In [7]:
entries = ["Restraints Used\n", "Smattery"]


def clean_occurrences(entries):

    actual_occurrences = ["Suicide (method)", "Suicide (attempt)", "Homicide", "Homicide Attempt", "Escape", "Escape Attempt",
                "Fire", "Serious Injury", "Battery", "Riot of Rebellion", "Sex Offense", "Assault on Staff",
                "Assault among Detainees", "Fighting among Detainees", "Restraints Used", "OC Spray Used", "Other (specify)"]
    
    if entries.is_empty():
        return ["Error, no occurrence found"]

    cleaned_list = []
    for term in entries:
        best_choice = None
        highest_jaro = 0.0
        for compare_term in actual_occurrences:
            jaro_score = jellyfish.jaro_similarity(term, compare_term)
            if jaro_score > highest_jaro:
                highest_jaro = jaro_score
                best_choice = compare_term

        if best_choice in ["Suicide (method)", "Suicide (attempt)", "Other (specify)"]:
            # Keep the original term (possibly has colon text, like "Other (specify): Fight")
            cleaned_list.append(term)
        else:
            if best_choice not in cleaned_list:
                cleaned_list.append(best_choice)

    return cleaned_list

In [9]:
all_names = set(names.words())

test_list = ["Deaunte Campbell","3/12/2002","10/13/2021"]

def get_inmate_names(table_list):
    inmate_list = []
    pattern_punctuation = r"[^a-zA-Z0-9/ ]"
    for item in table_list:
        cleaned_item = re.sub(pattern_punctuation, '', item)
        tokens = cleaned_item.strip().split()
        for token in tokens:
            token = token.capitalize()
            if token in all_names:
                inmate_list.append(cleaned_item)
                break
    return inmate_list

get_inmate_names(test_list)

[]

In [10]:
occurrence_check = samples_df.select(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))
samples_df = samples_df.with_columns(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))
samples_df = samples_df.with_columns(pl.col("Table Contents").map_elements(get_inmate_names).alias("Inmate Names"))

C:\Users\ganon\AppData\Local\Temp\ipykernel_31096\1834402618.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  occurrence_check = samples_df.select(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))
C:\Users\ganon\AppData\Local\Temp\ipykernel_31096\1834402618.py:2: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  samples_df = samples_df.with_columns(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))
C:\Users\ganon\AppData\Local\Temp\ipykernel_31096\1834402618.py:3: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  samples_df = samples_df.with_columns(pl.col

In [11]:
# Clean facility name:
def clean_facility_name(df):
    """Clean facility names using Polars expressions"""
    return df.with_columns(
        pl.col('Facility Name')
        # First, clean OCR artifacts
        .str.replace_all(r'\\n|\\\\|\\/', ' ') 
        .str.replace_all(r'\n', ' ')
        .str.replace_all(r'[\\\/]+', ' ')
        .str.replace_all(r'\s+', ' ') 
        .str.strip_chars()
        # Extract facility name after any variation of "Name:"
        .str.extract(r'cility Name:\s*(.+)', 1)
        # Additional cleanup of extracted names
        .str.replace_all(r'[^a-zA-Z\s\'-]', ' ')  # Keep only letters, spaces, apostrophes, hyphens
        .str.replace_all(r'\s+', ' ')  # Clean multiple spaces again
        .str.strip_chars()  # Final trim
        # Stop at any word ending in "ddres" and remove that word + everything after
        .str.replace_all(r'\s+(ddress|ddrace).*$', '')
        .str.replace_all(r"(?i)\bSheriff's\b", "Sheriffs")
        .str.to_titlecase()
        .alias('Cleaned Facility Name')
    )

samples_df = clean_facility_name(samples_df)

# Date cleanning
def clean_date_occurrence(df):
    return df.with_columns(
        pl.col("Date")
          # OCR cleanup (keep forward slashes)
          .str.replace_all(r"\\n|\\\\", " ")
          .str.replace_all(r"\n", " ")
          .str.replace_all(r"\s+", " ")
          .str.strip_chars()
          # Extract the MM/DD/YYYY text
          .str.extract(r"Date of Occurrence:\s*([0-9]{1,2}/[0-9]{1,2}/(20[0-9]{2}))", 1)
          # Parse & re‐format to zero‐padded MM/DD/YYYY
          .str.strptime(pl.Date, "%m/%d/%Y", strict=False)
          .alias("temp_date")
    ).with_columns(
        # Only keep dates between 2000 and 2025, otherwise set to null
        pl.when(pl.col("temp_date").dt.year().is_between(2000, 2025))
        .then(pl.col("temp_date").dt.strftime("%m/%d/%Y"))
        .otherwise(None)
        .alias("Cleaned Date")
    ).drop("temp_date")

samples_df = clean_date_occurrence(samples_df)

In [12]:
samples_df = samples_df.drop(["OCR_Confidence", "OCR_Word_Count", "OCR_Text_Detected", "Occurrence", "Facility Name", "Date"])

samples_df = samples_df.with_columns(pl.col("Cleaned Occurrences").list.join(",").alias("Cleaned Occurrences"))

samples_df = samples_df.with_columns(pl.col("Inmate Names").list.join(",").alias("Inmate Names"))

samples_df = samples_df.with_columns(pl.col("Table Contents").list.join(",").alias("Table Contents"))

In [13]:
samples_df = samples_df.select(["Report ID", "Facility Type", "Cleaned Facility Name", "Address", "Phone Number", "Cleaned Date", "Time of Day", "AM or PM", "Table Contents", "Inmate Names", "Cleaned Occurrences", "Injuries?", "Resulting Death?", 'Deceased Cause', 'Deceased Date and Time', 'Deceased on Suicide Watch', 'Deceased Reporter', 'Deceased Examined by Physician', 'Deceased Signs of Illness'])

In [14]:
samples_df = samples_df.select(["Report ID", "Cleaned Occurrences", "Facility Type", "Cleaned Facility Name", "Address", "Phone Number", "Cleaned Date", "Time of Day", "AM or PM", "Table Contents", "Inmate Names", "Injuries?", "Resulting Death?", 'Deceased Cause', 'Deceased Date and Time', 'Deceased on Suicide Watch', 'Deceased Reporter', 'Deceased Examined by Physician', 'Deceased Signs of Illness'])

In [15]:
test = samples_df.filter(
    pl.col("Cleaned Occurrences").str.contains("Rest")
)

In [ ]:
#test.write_excel("restraints_sample.xlsx")

In [163]:
test

Report ID,Cleaned Occurrences,Facility Type,Cleaned Facility Name,Address,Phone Number,Cleaned Date,Time of Day,AM or PM,Table Contents,Inmate Names,Injuries?,Resulting Death?,Deceased Cause,Deceased Date and Time,Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""Ponce, David 1-17_p1""","""Other (specify): Resisting ,Re…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""01/17/2021""","""_ Time of Occurrence: 193\ ""","""No filled box found""","""Ponce, David Julian,04/13/1999…","""Ponce David Julian""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""Kiger, Bonham 1-2_p1""","""Restraints Used""","""Error creating ROI: zero-lengt…","""St Clair County Sheriffs Depar…","""\ddress: 700 North 5"" Street B…","""‘elephone #: 618-277-3505 ""","""01/02/2021""","""me of Occurrence: 11:10 Oam. ""","""Error: less than two boxes fou…","""| 12/21/1995,| 01/02/2021,Ag B…","""Ag Bate""","""No injuries""","""No ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""Martin, Kaitlyn 1-10_p1""","""Restraints Used,Other (specify…","""County ""","""Tazewell County Jail""","""\ddress: 101 S. Capitol Pekin …","""_._ Telephone #: 309-478-5613 ""","""01/10/2020""","""_ Time of Occurrence: 0435 ""","""a.m. ""","""01/28/200 1,- o1/ior2021,mee e…","""""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""Whiting, Preston 1-5_p1""","""Restraints Used,Sther (spocify…","""Error creating ROI: text ROI i…","""Knox County Sheriffs Departmen…","""iddress: 152 S. Kellogg St. Ga…","""____ Telephone #: 309-345-3760…","""01/05/2021""","""wy wee _ Time of Occurrence: 1…","""pM. ""","""aera seater,Whiting, Preston S…","""Whiting Preston Scott""","""No injuries""","""No ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""Ledbetter, Michael 1-27_p1""","""Restraints Used""","""County ""","""Sangamon County Jail""","""Address: #1 Sheriffs Plaza Spr…","""__. Telephone #: 217-753-6886 ""","""01/27/2021""","""_ Time of Occurrence: 1305 ""","""p.m, ""","""“Michael Ledbetter,1 08/06/198…","""Michael Ledbetter""","""Yes, (briefly describe). Bumpe…","""Error: less than two boxes fou…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FOIA - UO July 2023_p486""","""Other (specify): Rest chair ,R…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""07/27/2023""","""_ Time of Occurrence: 9:4) ""","""p.m. ""","""Villa, Moises,12/26/2002,04/18…","""Villa Moises""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""FOIA - UO July 2023_p512""","""Restraints Used""","""County ""","""Peoria County Sheriffs Office""","""\ddress: 301 N. Maxwell Road P…","""__._- Telephone #: 309-697-784…","""07/29/2023""","""_ Time of Occurrence: 1730 ""","""p.m. ""","""Farmer, Sentaries,08/19/2004,0…","""""","""No injuries""","""Error creating ROI: zero-lengt…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""FOIA - UO July 2023_p521""","""Other (specify): ,Restraints U…","""County ""","""Peoria County Sheriffs Office""","""\ddress: 301 N. Maxwell Road P…","""__._- Telephone #: 309-697-784…","""07/29/2023""","""_ Time of Occurrence: 1130 ""","""a.m. ""","""Swartz, Henry,03/21/1996,02/27…","""Swartz Henry""","""No injuries""","""Error creating ROI: zero-lengt…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""


In [16]:
def clean_phone_number(df):
    pattern = r"#:"
    df = df.with_columns(
    pl.when(pl.col("Phone Number").str.contains(pattern))
    .then(pl.col("Phone Number").str.extract(r"#\s*(\S.*)"))
    .otherwise(pl.col("Phone Number"))
    .alias("Cleaned Phone Number")
    )

    return df

samples_df = clean_phone_number(samples_df)

In [17]:
def clean_time(df):
    pattern = r"Occurrence:"
    df = df.with_columns(
    pl.when(pl.col("Time of Day").str.contains(pattern))
    .then(pl.col("Time of Day").str.extract(r"e:\s*(\S.*)"))
    .otherwise(pl.col("Time of Day"))
    .alias("Cleaned Time of Day")
    )

    return df

samples_df = clean_time(samples_df)

In [18]:
def clean_phone_number(df):
    pattern = r"ss:"
    df = df.with_columns(
    pl.when(pl.col("Address").str.contains(pattern))
    .then(pl.col("Address").str.extract(r"s:\s*(\S.*)"))
    .otherwise(pl.col("Address"))
    .alias("Cleaned Address")
    )

    return df

In [19]:
def clean_address(df):
    pattern = r"ss:"
    df = df.with_columns(
    pl.when(pl.col("Address").str.contains(pattern))
    .then(pl.col("Address").str.extract(r"s:\s*(\S.*)"))
    .otherwise(pl.col("Address"))
    .alias("Cleaned Address")
    )

    return df

samples_df = clean_address(samples_df)

In [20]:
def get_zip_code(df):
    df = df.with_columns(
    pl.col("Cleaned Address").str.extract(r"(\d{5})").alias("Zip Code")
    )

    return df

samples_df = get_zip_code(samples_df)

In [21]:
def clean_injuries(df):
    pattern = r"ibe\)[;:]"
    df = df.with_columns(
    pl.when(pl.col("Injuries?").str.contains(pattern))
    .then(pl.col("Injuries?").str.extract(r"\)[;:]\s*(\S.*)"))
    .otherwise(pl.col("Injuries?"))
    .alias("Cleaned Injuries")
    )

    return df

samples_df = clean_injuries(samples_df)

In [211]:
df_sampled = samples_df.sample(n=100)

In [85]:
df_sampled

Report ID,Cleaned Occurrences,Facility Type,Cleaned Facility Name,Address,Phone Number,Cleaned Date,Time of Day,AM or PM,Table Contents,Inmate Names,Injuries?,Resulting Death?,Deceased Cause,Deceased Date and Time,Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness,Cleaned Phone Number,Cleaned Time of Day,Cleaned Address,Zip Code,Cleaned Injuries
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""FOIA - Feb 2024 UOs_p375""","""Suicide Attempt (method) State…","""County ""","""Peoria County""","""Address: 301 N. Maxwell Rd Peo…",""": 309-697-8515""","""02/27/2024""","""City State _ Time of Occurrenc…","""p.m. ""","""Holobaugh, Kenneth,05/08/1981,…","""Holobaugh Kenneth""","""No injuries""","""No ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 309-697-8515""","""21:45""","""301 N. Maxwell Rd Peoria IL 61…","""61604""","""No injuries"""
"""FOIA - UO August 2023_p450""","""Other (specify): Hospital trn ""","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…",""": 630-232-6840""","""08/22/2023""","""_ Time of Occurrence: 2:1. ""","""a.m. ""","""Stewart, Zanzibah,05/11/1978,1…","""Stewart Zanzibah""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 630-232-6840""","""2:1.""","""37W755 IL Route 38 Suite A Sai…","""60175""","""No injuries"""
"""FOIA - UO April 2024_p165""","""Fighting among Detainees""","""""","""Dupage County Correctional Cen…","""Address: 501 N County Farm rd …",""": 6304072200""","""04/11/2024""","""_ Time of Occurrence: 1400 ""","""p.m. ""","""weeee me,Dawson, Jeremiah,| Ag…","""Dawson Jeremiah, Turner Romain…","""Yes, (briefly describe): split…","""No ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 6304072200""","""1400""","""501 N County Farm rd Wheaton I…","""60187""","""splitlip __——s—s—“—ti—i—‘“‘“‘“…"
"""UO - FOIA January 2023_p100""","""Error, no occurrence found""","""Error: less than two boxes fou…","""Cook County Department Of Corr…","""Street City Chote Fin Cada """,""": 773.674.3474""","""01/09/2023""","""SERRE. WE RSS EE Pie Fama ew F…","""Check time, no AM or PM for Co…","""Ceddrell A Silas-Peterson,4/24…","""""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 773.674.3474""","""SERRE. WE RSS EE Pie Fama ew F…","""Street City Chote Fin Cada """,null,"""No injuries"""
"""UO - FOIA December 2022_p503""","""Battery""","""Count, ""","""Cook County Department Of Corr…","""Street City Ctate Fin Cade """,""": 773.674.3474""","""12/04/2022""","""SSPE. WE PSE EP we aor YW Fle …","""Check time, no AM or PM for Co…","""Cristian Zuniga,5/8/1997,10/14…","""Brian E Otero,Demetrius Flemin…","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 773.674.3474""","""SSPE. WE PSE EP we aor YW Fle …","""Street City Ctate Fin Cade """,null,"""No injuries"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FOIA - Sept 2024 UO Cook Co_p1…","""OC Spray Used""","""Error: less than two boxes fou…","""Cook County Department Of Corr…","""Aaqaress: S2IFUY Y- YSGHIVETHI…",""": 773.674.3474""","""09/16/2024""","""re_ QO, tataa AMawnme fac ti. …","""Check time, no AM or PM for Co…","""Richard Eames,1/5/1994,9/7/202…","""Richard Eames""","""No injuries""","""Error: less than two boxes fou…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 773.674.3474""","""re_ QO, tataa AMawnme fac ti. …","""S2IFUY Y- YSGHIVETHIG FAVOR MI…",null,"""No injuries"""
"""UO - FOIA December 2022_p264""","""ther (specify): Sick Person ""","""Error: less than two boxes fou…","""Lake County Adult Corrections""","""Address: 29S Martin Luther Kin…",""": 847-377-4099""","""12/23/2022""","""Cnty otate Time of Occurrence:…","""p.m. ""","""Lewis, Consuelo L171912,on 91 …","""Lewis Consuelo L171912""","""No i

In [23]:
count = samples_df.filter(
    pl.col("Cleaned Occurrences").str.contains("Error, no occurrence found")
)

In [ ]:
count

In [ ]:
samples_df = samples_df.with_columns(
    pl.when(pl.col("Cleaned Occurrences") == "")
    .then(pl.lit("Error, no occurrence found"))
    .otherwise(pl.col("Cleaned Occurrences"))
    .alias("Cleaned Occurrences")
)

def clean_other(df):
    pattern = r"specify"
    df = df.with_columns(
    pl.when(pl.col("Cleaned Occurrences").str.contains(pattern))
    .then(pl.col("Cleaned Occurrences").str.extract(r"\):\s*(\S.*)"))
    .otherwise(pl.col("Cleaned Occurrences"))
    .alias("Cleaned Occurrences")
    )

    return df

total_occurrence_result = (
    samples_df
    .with_columns(
        pl.col("Cleaned Occurrences").str.split(",").alias("Cleaned Occurrences")
    )
    .explode("Cleaned Occurrences")
    .filter(pl.col("Cleaned Occurrences").is_not_null())
    .select(
        pl.col("Cleaned Occurrences").value_counts(sort=True)
    )
    .unnest("Cleaned Occurrences")
    .head(12)
)

total_occurrence_result = clean_other(total_occurrence_result)

total_occurrence_result = total_occurrence_result.filter(~pl.col("Cleaned Occurrences").is_in(["Error", " no occurrence found"]))

total_occurrence_chart = alt.Chart(total_occurrence_result).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Occurrences:N", title="Occurrence").sort('-x')
).properties(
    title="Most Common Occurrences Across All Jails"
)

total_occurrence_chart = alt.Chart(total_occurrence_result).mark_bar(
    color="#54728d"
).encode(
    x=alt.X('count:Q', title='Number of Cases'),
    y=alt.Y('Cleaned Occurrences:N', sort='-x', title='Occurrence'),
    tooltip=[
        alt.Tooltip('Cleaned Occurrences:N', title='Occurrence'),
        alt.Tooltip('count:Q', format=',d', title='Case Count')
    ]
).properties(
    width=600,
    height=400,
    title='Most Common Occurrences Across All Jails'
)

# Footnote as a separate text chart (keep your note)
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Occurrences may be concurrent. Missing 2,027 Occurrences (7.3%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,
    color="gray"
).encode(
    x=alt.value(-50),
    y=alt.value(440),
    text="text:N"
)

# Layer chart and footnote, remove box
total_occurrence_chart = alt.layer(
    total_occurrence_chart,
    footnote
).configure_view(
    stroke=None
)

total_occurrence_chart

#Missing 2,027 Occurrences (7.3%)"

alt.LayerChart(...)

In [213]:
pattern = r"\bspecify\b"

other_occurrence_table = samples_df.filter(
    pl.col("Cleaned Occurrences").str.contains(pattern)
)
    
other_occurrence_table = clean_other(other_occurrence_table)

other_occurrence_table = (
    other_occurrence_table.select(
        pl.col("Cleaned Occurrences").value_counts(sort=True)
    )
    .unnest("Cleaned Occurrences")
    .head(10)
)

other_occurrence_chart = alt.Chart(other_occurrence_table).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Occurrences:N", title="Occurrence").sort('-x')
).properties(
    title="Most Common Other Occurrences Across All Jails"
)

In [214]:
other_occurrence_chart

alt.Chart(...)

In [171]:
samples_df = clean_other(samples_df)

In [25]:
top_prisons = (
    samples_df.select(
        pl.col("Cleaned Facility Name").value_counts(sort=True)
    )
    .unnest("Cleaned Facility Name")
    .head(11)
    )

top_prisons = top_prisons.filter(pl.col("Cleaned Facility Name") != "")

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Missing 1,119 County Names (4.33%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(-50),
    y=alt.value(230),  # just below the chart (chart height + some offset)
    text="text:N"
)

top_prisons_chart = alt.Chart(top_prisons).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    title="Occurrence Counts by Jails"
)

# Use layering to draw the footnote over/under the chart
top_prisons_chart = alt.layer(
    top_prisons_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

top_prisons_chart

alt.LayerChart(...)

In [26]:
top_counties = (
    samples_df
    .select(["Cleaned Facility Name", "Cleaned Occurrences"])
    .with_columns(
        pl.col("Cleaned Occurrences").str.split(",").alias("Cleaned Occurrences")
    )
    .explode("Cleaned Occurrences")
    .with_columns(
        pl.col("Cleaned Occurrences").str.strip_chars().alias("Cleaned Occurrences")
    )
    .filter(pl.col("Cleaned Occurrences").is_not_null())
)

top_counties = top_counties.join(
    top_counties
    .select(pl.col("Cleaned Facility Name").value_counts())
    .unnest("Cleaned Facility Name")  # converts struct to columns
    .rename({"count": "count"}),  # optional, but clarifies
    on="Cleaned Facility Name",
    how="left"
)

def get_county_occurrences(current_jail):
    filtered_df = top_counties.filter(pl.col("Cleaned Facility Name") == current_jail)
    current_jail_result = (
    filtered_df.select(
        pl.col("Cleaned Occurrences").value_counts(sort=True)
    )
    .unnest("Cleaned Occurrences")
    .head(12)
    )
    current_jail_result = current_jail_result.filter(~pl.col("Cleaned Occurrences").is_in(["Error", "no occurrence found"]))
    current_jail_chart = alt.Chart(current_jail_result).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Occurrences:N", title="Occurrence").sort('-x')
    ).properties(
        title=current_jail
    )
    return current_jail_chart

def get_county_occurrences_percent(current_jail):
    filtered_df = top_counties.filter(pl.col("Cleaned Facility Name") == current_jail)
    current_jail_result = (
        filtered_df.select(
            pl.col("Cleaned Occurrences").value_counts(sort=True)
        )
        .unnest("Cleaned Occurrences")
        .filter(~pl.col("Cleaned Occurrences").is_in(["Error", "no occurrence found"]))
    )
    total_count = current_jail_result["count"].sum()
    current_jail_result = current_jail_result.with_columns(
        (pl.col("count") / total_count * 100).alias("percentage")
    )
    current_jail_result = current_jail_result.sort("count", descending=True).head(10)
    current_jail_chart = alt.Chart(current_jail_result.to_pandas()).mark_bar().encode(
        x=alt.X("percentage:Q", title="Percentage"),
        y=alt.Y("Cleaned Occurrences:N", title="Occurrence").sort('-x')
    ).properties(
        title=current_jail
    )

    return current_jail_chart

cook_county = get_county_occurrences("Cook County Department Of Corrections")
cook_county_percent = get_county_occurrences_percent("Cook County Department Of Corrections")

kane_county = get_county_occurrences("Kane County Sheriffs Office")
kane_county_percent = get_county_occurrences_percent("Kane County Sheriffs Office")

madison_county = get_county_occurrences("Madison County Sheriffs Office")
madison_county_percent = get_county_occurrences_percent("Madison County Sheriffs Office")

lake_county = get_county_occurrences("Lake County Jail")
lake_county_percent = get_county_occurrences_percent("Lake County Jail")

peoria = get_county_occurrences("Peoria County Sheriffs Office")
peoria_percent = get_county_occurrences_percent("Peoria County Sheriffs Office")


In [27]:
cook_county_percent

alt.Chart(...)

In [28]:
kane_county_percent

alt.Chart(...)

In [180]:
madison_county_percent

alt.Chart(...)

In [181]:
lake_county_percent

alt.Chart(...)

In [182]:
peoria_percent

alt.Chart(...)

In [29]:
pattern = r"\bRestraints\b"

restraints_table = top_counties.filter(
    pl.col("Cleaned Occurrences").str.contains(pattern)
)

restraints_table = (
    restraints_table.select(
        pl.col("Cleaned Facility Name").value_counts(sort=True)
    )
    .unnest("Cleaned Facility Name")
    .head(21)
)

restraints_table = restraints_table.filter(pl.col("Cleaned Facility Name") != "")

restraints_chart = alt.Chart(restraints_table).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    title="Top Jails for Restraint Usage"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Missing 1,119 County Names (4.33%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(0),
    y=alt.value(430),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
restraints_chart = alt.layer(
    restraints_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

In [30]:
restraints_chart

alt.LayerChart(...)

In [32]:
restraints_table = restraints_table.filter(pl.col("Cleaned Facility Name") != "null").join(top_counties, on="Cleaned Facility Name")

In [33]:
restraints_table = restraints_table.filter(
    pl.col("Cleaned Occurrences") == "Restraints Used"
)

restraints_table = restraints_table.unique()

In [34]:
restraints_perc_chart = alt.Chart(restraints_table.filter(pl.col("count_right") >= 100).with_columns(
    ((pl.col("count") / pl.col("count_right")).alias("perc")*100)
)).mark_bar().encode(
    x=alt.X("perc:Q", title="Percentage"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    title="Top Jails for Restraint Usage as a Percent of Reports"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Only Jails with over 100 Reports Total"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(0),
    y=alt.value(440),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
restraints_perc_chart = alt.layer(
    restraints_perc_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

In [35]:
restraints_perc_chart

alt.LayerChart(...)

In [36]:
pattern = r"\bFighting among Detainees\b"

fighting_table = top_counties.filter(
    pl.col("Cleaned Occurrences").str.contains(pattern)
)

fighting_table = (
    fighting_table.select(
        pl.col("Cleaned Facility Name").value_counts(sort=True)
    )
    .unnest("Cleaned Facility Name")
    .head(11)
)

fighting_table_table = fighting_table.filter(pl.col("Cleaned Facility Name") != "")

fighting_chart = alt.Chart(fighting_table).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    title="Top Jails for Fighting among Detainees"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Missing 214 Jail Names (12%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(0),
    y=alt.value(250),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
fighting_chart = alt.layer(
    fighting_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

In [37]:
fighting_table

Cleaned Facility Name,count
str,u32
"""Cook County Department Of Corr…",4727
"""Madison County Sheriffs Office""",451
"""Kane County Sheriffs Office""",171
"""Lake County Jail""",164
"""Sangamon County Jail""",143
…,…
"""Winnebago County Jail""",73
"""Cook County Department Of Corr…",67
"""Dupage County Jail""",60


In [38]:
fighting_chart

alt.LayerChart(...)

In [39]:
fighting_table = fighting_table.filter(pl.col("Cleaned Facility Name") != "null").join(top_counties, on="Cleaned Facility Name")

In [40]:
fighting_table = fighting_table.filter(
    pl.col("Cleaned Occurrences") == "Restraints Used"
)

fighting_table = fighting_table.unique()

fighting_perc_chart = alt.Chart(fighting_table.filter(pl.col("count_right") >= 100).with_columns(
    ((pl.col("count") / pl.col("count_right")).alias("perc")*100)
)).mark_bar().encode(
    x=alt.X("perc:Q", title="Percentage"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    title="Top Jails for Restraint Usage as a Percent of Reports"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Only Jails with over 100 Reports Total"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(0),
    y=alt.value(230),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
fighting_perc_chart = alt.layer(
    fighting_perc_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

In [41]:
fighting_perc_chart

alt.LayerChart(...)

In [42]:
top_people = (
    samples_df.select(
        pl.col("Inmate Names").value_counts(sort=True)
    )
    .unnest("Inmate Names")
    .head(10)
    )

case_study = samples_df.filter(
    pl.col("Inmate Names") == "Ordonez Robert F"
)

In [43]:
case_study

Report ID,Cleaned Occurrences,Facility Type,Cleaned Facility Name,Address,Phone Number,Cleaned Date,Time of Day,AM or PM,Table Contents,Inmate Names,Injuries?,Resulting Death?,Deceased Cause,Deceased Date and Time,Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness,Cleaned Phone Number,Cleaned Time of Day,Cleaned Address,Zip Code,Cleaned Injuries
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""April 2021 FOIA Request_p67""","""Other (specify): Rest Chair ,O…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""04/30/2021""","""_ Time of Occurrence: 040! ""","""No filled box found""","""Ordonez, Robert F,08/12/1981,0…","""Ordonez Robert F""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 630-232-6840""","""040!""","""37W755 IL Route 38 Suite A Sai…","""60175""","""No injuries"""
"""April 2021 FOIA Request_p68""","""Other (specify): Rest Chair ,R…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""04/28/2021""","""_ Time of Occurrence: 034 ""","""No filled box found""","""Ordonez, Robert F.,08/12/1981,…","""Ordonez Robert F""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 630-232-6840""","""034""","""37W755 IL Route 38 Suite A Sai…","""60175""","""No injuries"""
"""April 2021 FOIA Request_p69""","""Other (specify): Rest Chair ,R…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""04/27/2021""","""_ Time of Occurrence: 205: ""","""No filled box found""","""Ordonez, Robert F.,08/12/1981,…","""Ordonez Robert F""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 630-232-6840""","""205:""","""37W755 IL Route 38 Suite A Sai…","""60175""","""No injuries"""
"""April 2021 FOIA Request_p70""","""Other (specify): Rest Chair ,R…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""04/26/2021""","""_ Time of Occurrence: 143: ""","""No filled box found""","""Ordonez, Robert F.,08/12/1981,…","""Ordonez Robert F""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 630-232-6840""","""143:""","""37W755 IL Route 38 Suite A Sai…","""60175""","""No injuries"""
"""April 2021 FOIA Request_p71""","""Other (specify): Rest Chair ,R…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""04/25/2021""","""_ Time of Occurrence: 212: ""","""No filled box found""","""Ordonez, Robert F.,08/12/1981,…","""Ordonez Robert F""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 630-232-6840""","""212:""","""37W755 IL Route 38 Suite A Sai…","""60175""","""No injuries"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Binder1_p279""","""Other (specify): Rest Chair ,R…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""06/22/2021""","""_ Time of Occurrence: 1210 hr ""","""No filled box found""","""Ordonez, Robert F.,08/12/1981,…","""Ordonez Robert F""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 630-232-6840""","""1210 hr""","""37W755 IL Route 38 Suite A Sai…","""60175""","""No injuries"""
"""Binder1_p280""","""Other (specify): Rest Chair ,R…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""06/23/2021""","""_ Time of Occurrence: 0342 hr ""","""No filled box found""","""Ordonez, Robert F.,08/12/1981,…","""Ordonez

In [113]:
#samples_df.write_excel("7_10_sample.xlsx")

To-Do:
- Breakdown of largest counties